# Error Handling in Azure Data Factory (ADF)

## What is Error Handling?

**Error Handling** in Azure Data Factory (ADF) is the process of **detecting, managing, logging, and recovering from failures** that occur during pipeline execution.

Without proper error handling:

* Pipelines stop unexpectedly.
* You don't know which activity failed.
* No audit trail is created.
* Notifications are not sent.
* Failed files may be processed again incorrectly.

With proper error handling:

* Errors are logged.
* Notifications are sent.
* Failed files are isolated.
* Pipelines can retry or continue based on business rules.

---

# Why is Error Handling Important?

Imagine an ETL pipeline:

```text
SQL Server
     │
     ▼
Copy Activity
     │
     ▼
Databricks Notebook
     │
     ▼
Stored Procedure
     │
     ▼
Power BI
```

If the **Copy Activity** fails because the source database is unavailable:

* Data isn't copied.
* Databricks shouldn't run.
* Audit tables should record the failure.
* Support teams should be notified.

Error handling manages this entire process.

---

# Types of Errors in ADF

## 1. Source Errors

Examples:

* Source database is down.
* File not found.
* Invalid credentials.
* Network timeout.

Example:

```text
Blob Storage

↓

File Missing

↓

Copy Activity Failed
```

---

## 2. Data Errors

Examples:

* Invalid date format
* NULL in NOT NULL column
* Data type mismatch

Example

```text
Age

↓

ABC

↓

INT Column

↓

Failed
```

---

## 3. Destination Errors

Examples

* Table does not exist
* Disk full
* Permission denied

---

## 4. Network Errors

Examples

* Connection timeout
* VPN disconnected
* DNS issue

---

## 5. Business Rule Errors

Example

```text
Sales Amount

↓

-500

↓

Invalid

↓

Fail Activity
```

---

# Error Handling Mechanisms in ADF

ADF provides multiple ways to handle errors.

1. Activity Dependency Conditions
2. Retry Policy
3. Try-Catch Pattern
4. Fail Activity
5. Stored Procedure Logging
6. Web Activity Notifications
7. Custom Error Pipelines

---

# 1. Activity Dependency Conditions

Every activity can have different dependency paths:

```text
Succeeded

Failed

Skipped

Completed
```

Instead of only connecting activities on success, you can specify what should happen when an activity fails.

---

### Example

```text
Copy Activity
     │
 ┌───┴────────────┐
 │                │
Succeeded       Failed
 │                │
 ▼                ▼
Data Flow     Send Email
```

If the Copy Activity fails, the pipeline sends an email instead of continuing.

---

# 2. Retry Policy

ADF can retry an activity automatically.

Suppose SQL Server is temporarily unavailable.

Instead of failing immediately:

```text
Attempt 1

↓

Failed

↓

Retry

↓

Attempt 2

↓

Success
```

Example configuration:

* Retry count: **3**
* Retry interval: **60 seconds**

This is useful for transient failures such as network issues or temporary database unavailability.

---

# 3. Try-Catch Pattern

ADF doesn't have a dedicated **try-catch** activity, but you can implement the same behavior using activity dependencies.

### Pipeline

```text
Copy Activity
      │
      ├────────────── Success ─────► Data Flow
      │
      └────────────── Failure ─────► Log Error
                                      │
                                      ▼
                                  Send Email
```

This pattern is widely used in production pipelines.

---

# 4. Fail Activity

You can intentionally fail a pipeline when business rules are violated.

Example:

```text
Order Amount

↓

-100

↓

Fail Activity

↓

Pipeline Failed
```

Message:

```text
Negative sales amount detected.
```

This clearly identifies why the pipeline stopped.

---

# 5. Logging Errors

Enterprise projects almost always maintain an audit or logging table.

Example:

| Pipeline   | Activity | Status | Error         | Time     |
| ---------- | -------- | ------ | ------------- | -------- |
| DailySales | Copy     | Failed | Login Timeout | 10:15 AM |

ADF uses a **Stored Procedure Activity** or **Script Activity** to insert error details.

Example SQL:

```sql
INSERT INTO AuditLog
(
PipelineName,
ActivityName,
Status,
ErrorMessage,
ExecutionTime
)
VALUES
(
'DailySales',
'Copy Activity',
'Failed',
'Login Timeout',
GETDATE()
);
```

---

# 6. Notifications

After logging the error, notify the support team.

Pipeline

```text
Copy Activity

↓

Failed

↓

Web Activity

↓

Logic App

↓

Email

↓

Support Team
```

Email Example

```text
Subject:

ADF Pipeline Failed

Pipeline:
Daily Sales

Activity:
Copy Activity

Reason:
SQL Server Login Timeout
```

---

# 7. Error Folder Pattern

When processing files, move failed files to a dedicated folder.

```text
Incoming

↓

Copy Activity

↓

Success

↓

Processed

OR

Failure

↓

Error Folder
```

Folder structure:

```text
/raw

/processed

/error

/archive
```

This prevents repeatedly processing bad files.

---

# 8. Child Pipeline Error Handling

Large solutions often use a **Master Pipeline**.

```text
Master Pipeline

↓

Customer Pipeline

↓

Sales Pipeline

↓

Finance Pipeline
```

If the Sales Pipeline fails:

```text
Execute Pipeline

↓

Failed

↓

Log Error

↓

Continue or Stop
```

The master pipeline decides whether to stop the workflow or continue with other pipelines.

---

# Real-Time Enterprise Example

### Daily Retail ETL

```text
SQL Server

↓

Get Metadata

↓

Copy Activity

↓

Databricks Notebook

↓

Stored Procedure

↓

Power BI
```

### Error Scenario

SQL Server is down.

Pipeline:

```text
Copy Activity

↓

Failed

↓

Retry (3 Times)

↓

Still Failed

↓

Stored Procedure

(Log Error)

↓

Web Activity

(Send Email)

↓

Fail Activity

↓

Pipeline Ends
```

---

# Common Error Handling Architecture

```text
Source

↓

Copy Activity

↓

Success

↓

Data Flow

↓

Stored Procedure

↓

Success Email



Failure

↓

Retry

↓

Log Error

↓

Move File to Error Folder

↓

Send Email

↓

Fail Pipeline
```

---

# Best Practices

* Configure retry policies for transient failures.
* Log every failure with pipeline name, activity, timestamp, and error message.
* Use Logic Apps or Web Activity to notify support teams.
* Move invalid input files to an error folder.
* Parameterize error messages where possible.
* Keep audit and logging separate from business data.
* Use child pipelines to centralize common error-handling logic.

---

# Interview Questions

### 1. What is error handling in ADF?

**Answer:** Error handling is the process of detecting, managing, logging, and responding to pipeline failures so that issues are traceable and recovery actions can be performed.

---

### 2. How do you implement error handling in ADF?

A typical implementation includes:

1. Configure retry policies.
2. Use **Failed** dependency paths.
3. Log failures to an audit table.
4. Send notifications using Web Activity or Logic Apps.
5. Optionally move failed files to an error folder.
6. End the pipeline with a **Fail Activity** if necessary.

---

### 3. How do you send emails when a pipeline fails?

ADF doesn't send emails directly. The common approach is:

```text
Pipeline Failure

↓

Web Activity

↓

Logic App

↓

Email
```

The Logic App formats and sends the email to the required recipients.

---

### 4. What is the difference between Retry and Failed dependency?

| Retry                                   | Failed Dependency                             |
| --------------------------------------- | --------------------------------------------- |
| Retries the same activity automatically | Executes another activity after failure       |
| Used for temporary issues               | Used for logging, notifications, cleanup      |
| Configured in activity settings         | Configured via activity dependency conditions |

---

### 5. How do enterprise ADF projects handle errors?

A common production pattern is:

```text
ADF Pipeline

↓

Retry

↓

If Failed

↓

Log Error (SQL Audit Table)

↓

Move File to Error Folder

↓

Call Logic App (Email/Teams Notification)

↓

Fail Activity

↓

Pipeline Ends
```

This pattern ensures failures are visible, recoverable, and auditable while preventing downstream processes from running on incomplete or invalid data.
